This notebook is adapted from the [Dataflowr Linear Regression notebook](https://github.com/dataflowr/notebooks/blob/master/Module2/02b_linear_reg.ipynb) that you can visit to go further than this version.

To access this notebook on colab: https://colab.research.google.com/drive/1MBXNZ0vQzjGRyIVbiD1VnlLQWWzk5vds?usp=sharing.

# Automatic differentiation and application to Linear Regression

The goal of this notebook is to manipulate PyTorch tensors and automatic differentiation, then implement linear regression three ways:
- Closed-form (normal equation),
- Manual gradients (by hand),
- Using PyTorch automatic differentation.

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline
import torch
import numpy as np

## Autograd basics

When executing tensor operations, PyTorch can automatically construct on-the-fly the graph of operations to compute the gradient of any quantity with respect to any tensor involved.

To be more concrete, we introduce the following example: we consider parameters $w\in \mathbb{R}$ and $b\in \mathbb{R}$ with the corresponding function:
$$
\ell = \left(\exp(wx+b) - y^* \right)^2
$$

Our goal here, will be to compute the following partial derivatives:
$$
\frac{\partial \ell}{\partial w} \quad \textrm{and} \quad \frac{\partial \ell}{\partial b}.
$$

You can decompose this function as a composition of basic operations. This is call the forward pass on the graph of operations.
![backprop1](https://dataflowr.github.io/notebooks/Module2/img/backprop1.png)

> **Q1.** Create three scalar tensors `x`, `w`and `b`with values 0.5, 2 and 0.5.

In [ ]:
# YOUR CODE HERE

A `tensor` has a Boolean attribute `requires_grad`, set to `False` by default, which states if PyTorch should build the graph of operations so that gradients with respect to it can be computed.

> **Q2.** Since we want to take derivative with respect to $w$ and $b$, use the in-place operation to make it True.

In [ ]:
# YOUR CODE HERE

print(x, w, b)

> **Q3.** Complete the function below to compute $\ell(y, \hat{y}) = (\exp(w*x+b) `- y^\star)^2$. Test to evaluate your function at $y^\star=1.2$.

In [ ]:
def fun(x, w, b, ystar):
    # YOUR CODE HERE
    return loss


l = fun(x, w, b, ystar)
print(l)

The *forward pass* is done: we evaluated our loss with the current parameters $w$ and $b$ for a given $x$ and $y^\star$.

> **Q4.** Check that `l` can be used to compute gradient. Then, compute the gradient using automatic differentation. In pytorch, you can do it using [`tensor.backward()`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html). Note that all arguments are optional.

In [ ]:
# YOUR CODE HERE

> **Q5.** `tensor.backward()` accumulates the gradients in the `grad` fields of tensors. Print the values of the gradient with respect to $w$ and $b$. 

In [ ]:
# YOUR CODE HERE

By default, backward deletes the computational graph when it is used so that you will get an error if you execute the `backward()` operation again.

The gradients must be set to zero manually. Otherwise they will cumulate across several .backward() calls. This accumulating behavior is desirable in particular to compute the gradient of a loss summed over several “mini-batches,” or the gradient of a sum of losses.

In [ ]:
# Manually zero the gradients
w.grad.data.zero_()
b.grad.data.zero_()

# Evaluate the function
l = fun(x, w, b, ystar)

# Compute the gradients
l.backward(retain_graph=True)   # retain_graph=True to keep the graph for further backward calls
l.backward()                    # Now it will work since we retained the graph in the previous call

# Print the gradients
print(w.grad)       # Should be double the previous value
print(b.grad)       # Should be double the previous value

> **Q6.** Apply the chain rule to get the analytical derivative and evaluate it at our values of $w$, $b$, $x$ and $y^\star$.

<details>
  <summary><b>Show solution (if stuck)</b></summary>
  <img src="https://dataflowr.github.io/notebooks/Module2/img/backprop2.png"
       alt="Backprop diagram" style="max-width:100%; height:auto;">
</details>

> **Q7.** Implement it in torch, and compare the analytical derivative to autograd.

In [ ]:
# YOUR CODE of the analytical derivative here

## Linear Regression

The linear regression model for feature vectors in 2 dimensions ${\bf x} = [x^1, x^2]^\mathrm{T}$ reads
$$
f_{\theta}(x) = \hat{y}_i = w^1 x^1_t + w^2 x^2_i + b, \quad i\in\{1,\dots,n\}
$$

Our task in the training phase is to recover the weights $w^1, w^2$ and the bias $b$ given the data $({\bf x}_i,y_i)_{i\in\{1,\dots,n\}}$.

In order to do so, we choose the squared loss
$$\ell(y, \hat{y}) = (y-\hat{y})^2,$$
and minimize the *empirical risk*, that is we solve the following optimization problem
$$\underset{w^1,w^2,b}{\operatorname{argmin}} \sum_{i=1}^{n} \ell\left(y_i, \hat{y}_i\right),$$
where $w^1$, $w^2$ and $b$ are hidden in $\hat{y}_i$.

Let us consider a problem with $n=30$ data generated uniformly in $[0, 1]$. We construct the target $y_i^\star$ for each $\bf{x}_i$ as $y^\star_i = {\bf w^\star}^T{\bf x}_t+b^\star$.

In [ ]:
n = 30
d = 2
g = torch.Generator().manual_seed(32)
X = torch.rand(n, d, generator=g)       # Feature matrix

w_star = torch.tensor([2., -3.])
b_star = torch.tensor([1.])
y = X @ w_star + b_star                 # Target vector
y.unsqueeze_(1)

print(X[0:5])
print(y[0:5])

Below is a code to plot the dataset in the $(x^1, x^2, y^\star)$ space.

In [5]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D

def plot_figs(fig_num, elev, azim, x, y, weights, bias):
    fig = plt.figure(fig_num, figsize=(4, 3))
    plt.clf()
    ax = Axes3D(fig, elev=elev, azim=azim)
    ax.scatter(x[:, 0], x[:, 1], y)
    ax.plot_surface(np.array([[0, 0], [1, 1]]),
                    np.array([[0, 1], [0, 1]]),
                    (np.dot(np.array([[0, 0, 1, 1],
                                          [0, 1, 0, 1]]).T, weights) + bias).reshape((2, 2)),
                    alpha=.5)
    ax.set_xlabel('x_1')
    ax.set_ylabel('x_2')
    ax.set_zlabel('y')
    
def plot_views(x, y, w, b):
    #Generate the different figures from different views
    elev = 24.5
    azim = -110
    plot_figs(1, elev, azim, x, y, w, b[0])

    plt.show()

In [ ]:
plot_views(X.numpy(), y.numpy(), w_star.numpy(), b_star.numpy())

Now we have $X\in\mathbb{R}^{n\times d}$ and $\bf{y}$ containing the targets. Let's train a linear regression model on these data.

> **Q8.** Explain geometrically what linear regression optimization is achieving.

### Closed-form solution (Normal Equation)

> **Q9.** Remind the solution of the linear regression minimization problem.

> **Q10.** implement it using Torch tensor multiplication. You may need the Torch inverse function [`torch.linalg.inv`](https://docs.pytorch.org/docs/stable/generated/torch.linalg.inv.html).

In [ ]:
# YOUR CODE HERE

### Tensors and by-hand gradients

In vector form, we define
$$
\hat{y}_t = {\bf w}^T{\bf x}_t+b
$$
and we want to minimize the risk given by
$$
R(X, w^1, w^2, b) = \sum_i\underbrace{\left(\hat{y}_i-y_i \right)^2}_{\ell_i}.
$$

> **Q11.** To minimize the loss using gradient descent, first express the gradient of each $\ell_i$ with respect to each parameter, $\frac{\partial{\ell_i}}{\partial w^1}$, $\frac{\partial{\ell_i}}{\partial w^2}$, and $\frac{\partial{\ell_i}}{\partial b}$.

Note that the actual gradient of the loss is given by summing over all training examples
$$
\frac{\partial{R}}{\partial w^1} =\sum_i \frac{\partial{\ell_i}}{\partial w^1},\quad
\frac{\partial{R}}{\partial w^2} =\sum_i \frac{\partial{\ell_i}}{\partial w^2},\quad
\frac{\partial{R}}{\partial b} =\sum_i \frac{\partial{\ell_i}}{\partial b}
$$

For one epoch, **(Batch) Gradient Descent** updates the weights and bias as follows:
$$w^1_{new}=w^1_{old}-\alpha\frac{\partial{\ell}}{\partial w^1}$$
$$w^2_{new}=w^2_{old}-\alpha\frac{\partial{\ell}}{\partial w^2}$$
$$b_{new}=b_{old}-\alpha\frac{\partial{\ell}}{\partial b},$$
and then we run several epochs.

In [ ]:
# Randomly initialize learnable weights and bias
g = torch.Generator().manual_seed(32)       # Fix the seed for reproducibility
w_init = torch.rand(2, generator=g)
b_init = torch.rand(1, generator=g)

w_t = w_init.clone()
w_t.unsqueeze_(1)
b_t = b_init.clone()
b_t.unsqueeze_(1)
print("initial values of the parameters:", w_t, b_t )

> **Q12.** Complete the following functions computing the forward pass and the total loss function (empirical risk) over the training data $X$. The gradient is already implement and returns $\partial R / \partial w$ and $\partial R / \partial b$.

In [ ]:
# Our model forward pass
def forward_t(X):
    # YOUR CODE HERE
    return y_pred

# Loss function
def loss(X, y):
    # YOUR CODE HERE
    return loss

# Compute gradient
def gradient(X, y):  # d_loss/d_w, d_loss/d_b
    dRdw = 2 * (X.T @ (X @ w_t + b_t - y))
    dRdb = 2 * (X @ w_t + b_t - y).sum()
    return dRdw, dRdb

> **Q13.** Complete the following code to perform (batch) gradient descent using your implementations.

In [ ]:
# Reset the parameters before the loop
w_t = w_init.clone()
w_t.unsqueeze_(1)
b_t = b_init.clone()
b_t.unsqueeze_(1)

learning_rate = 1e-2
N_EPOCHS = 50
for epoch in range(N_EPOCHS):
    # Compute loss and gradient
    l_t = loss(X, y)
    grad_w, grad_b = gradient(X, y)
    
    # YOUR CODE HERE to update w_t and b_t
    
    # Print every 100 epochs
    if epoch%100 == 0 or epoch == N_EPOCHS-1:
        print("progress:", "epoch:", epoch, "loss", l_t)

# After training
print("Estimation of the parameters:", w_t, b_t)

> **Q14.** Vary the number of epochs and the learning rate. Comment.

### Torch Autograd

Now we will use the autograd facilities of Torch to compute the gradient descent without explicitly calculate it ourselves.

In [27]:
# Setting requires_grad=True indicates that we want to compute gradients with
# respect to these Tensors during the backward pass.
w_v = w_init.clone().unsqueeze(1)
w_v.requires_grad_(True)
b_v = b_init.clone().unsqueeze(1)
b_v.requires_grad_(True)
print("initial values of the parameters:", w_v.data, b_v.data )

initial values of the parameters: tensor([[0.8757],
        [0.2721]]) tensor([[0.4141]])


Training loop with autograd.

In [28]:
N_EPOCHS = 500
learning_rate = 1e-2

for epoch in range(N_EPOCHS):
    y_pred = X @ w_v + b_v
    loss = (y_pred - y).pow(2).sum()
    
    # Use autograd to compute the backward pass. This call will compute the
    # gradient of loss with respect to all Variables with requires_grad=True.
    # After this call w.grad and b.grad will be tensors holding the gradient
    # of the loss with respect to w and b respectively.
    loss.backward()
    
    # Update weights using gradient descent. For this step we just want to mutate
    # the values of w_v and b_v in-place; we don't want to build up a computational
    # graph for the update steps, so we use the torch.no_grad() context manager
    # to prevent PyTorch from building a computational graph for the updates
    with torch.no_grad():
        w_v -= learning_rate * w_v.grad
        b_v -= learning_rate * b_v.grad
    
    # Manually zero the gradients after updating weights
    # otherwise gradients will be accumulated after each .backward()
    w_v.grad.zero_()
    b_v.grad.zero_()
    
    if epoch%100 == 0 or epoch == N_EPOCHS-1:
        print("progress:", "epoch:", epoch, "loss", loss.item())

# After training
print("estimation of the parameters:", w_v.data, b_v.data.t() )

progress: epoch: 0 loss 31.695507049560547
progress: epoch: 100 loss 0.023344870656728745
progress: epoch: 200 loss 0.00011516775703057647
progress: epoch: 300 loss 5.711948460884742e-07
progress: epoch: 400 loss 2.8633095894292637e-09
progress: epoch: 499 loss 3.4926728176287725e-11
estimation of the parameters: tensor([[ 2.0000],
        [-3.0000]]) tensor([[1.0000]])


## Remark

This problem can be solved in 3 lines of code using the [least square](https://docs.pytorch.org/docs/stable/generated/torch.linalg.lstsq.html) function from Torch `torch.linalg.lstsq`.

In [29]:
Xb = torch.cat((X, torch.ones(n).unsqueeze(1)),1)
LR = torch.linalg.lstsq(Xb, y)
LR.solution

tensor([[ 2.0000],
        [-3.0000],
        [ 1.0000]])